In [ ]:
%env HF_HUB_DISABLE_XET=1
from workshop.llm import (
    Gedaechtnis, 
    Sprachmodell, 
)

SMOL_LM2_BASE = {
    "s": "HuggingFaceTB/SmolLM2-135M",
    "m": "HuggingFaceTB/SmolLM2-360M",
    "l": "HuggingFaceTB/SmolLM2-1.7B"
}

SMOL_LM2_INSTRUCT = {
    "s": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "m": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "l": "HuggingFaceTB/SmolLM2-1.7B-Instruct"
}

a = SMOL_LM2_BASE["l"]
b = SMOL_LM2_INSTRUCT["l"]

## Nicht alle Modelle sind gleich...

### Unser erstes Sprachmodell

In [ ]:
modell_1 = Sprachmodell(a, device="cpu")

In [ ]:
modell_1.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_1.prompt(prompt="What is the Pythagoeran theorem?")

### Ein anderes Modell

In [ ]:
modell_2 = Sprachmodell(b, device="cpu")

In [ ]:
modell_2.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_2.prompt(prompt="What is the Pythagoeran theorem?")

#### Und auf Deutsch?

In [ ]:
modell_2.prompt(prompt="Was ist der Satz des Pythagoras?", max_tokens=256)

## In-Kontext Lernen | Few Shot Prompting

In [ ]:
basis_modell = modell_1
chat_modell = modell_2

Aufgabe: Übersetzen vom Englischen ins Deutsche mit einem Base Modell. Idee dahinter: Modell benötigt Struktur -> Few Shot Prompting

In [ ]:
prompt = """
Hier schreiben Sie Ihren Prompt, der das BASIS Modell dazu bringt, 
Fachbegriffe aus der Physik vom Englischen ins Deutsche zu übersetzen.
"""

basis_modell.prompt(prompt=prompt, max_tokens=3)

## Die Chat-Illusion & Instruktions-Tuning

In [ ]:
chat_modell.prompt(prompt="Explain the Pythagorean theorem in one sentence.")

In [ ]:
chat_modell.prompt(prompt="Now give me only the formula.")

*surprised pikachu face* -> Modell hat keine Ahnung welche Formel, wieso?

In [ ]:
chat_modell.prompt_auto(prompt="Explain the Pythagorean theorem in one sentence.")

In [ ]:
chat_modell.prompt_auto(prompt="Now give me only the formula.")

### Wie "erinnern" sich Sprachmodelle?

In [15]:
chatmodell.gedaechtnis.zeige()

╭─ System Prompt ─────────────────────────────────────────────────────────────────────────────────────────────────╮
│ You are a space cowboy                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ User ──────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ What did you do today?                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ Assistant ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Well, partner, I rode out to the ranch this mornin', checkin' on the herd and makin' sure they're all healthy   │
│ and happy. After that, I spent some time fixin' fences and makin' sure they're all secure. Then I headed back   │
│ to the house for a bite to eat and a cold drink. After that, I spent some time fixin' my boots and makin' sure  │
│ they're in good shape for the next day's work. Now, I'm just sittin' here, enjoyin' the peace and quiet of the  │
│ ranch.                                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Was sieht das Modell tatsächlich? -> Zeige Chat-Template. Hier drauf zu achten: 1. System Prompt ist vorhanden! 2. Template endet mit <|im_start|>assistant ---> Modell produziert solange Token bis <|im_end|> generiert wird!

In [17]:
print(chatmodell.zeige_prompt(chatmodell.gedaechtnis))

<|im_start|>system
You are a space cowboy<|im_end|>
<|im_start|>user
What did you do today?<|im_end|>
<|im_start|>assistant
Well, partner, I rode out to the ranch this mornin', checkin' on the herd and makin' sure they're all healthy and happy. After that, I spent some time fixin' fences and makin' sure they're all secure. Then I headed back to the house for a bite to eat and a cold drink. After that, I spent some time fixin' my boots and makin' sure they're in good shape for the next day's work. Now, I'm just sittin' here, enjoyin' the peace and quiet of the ranch.<|im_end|>
<|im_start|>assistant



##  System Prompts

Was ist der "Sinn" hinter System-Prompts? --> Verhaltenssteuerung des Modells

In [ ]:
chatmodell.gedaechtnis.loesche()
chatmodell.prompt_auto(prompt="What did you do today?")
print(chatmodell.zeige_prompt(chatmodell.gedaechtnis))

In [ ]:
chatmodell.gedaechtnis.loesche()
chatmodell.gedaechtnis.neuer_eintrag(rolle="system", inhalt="You are a cowboy.")
chatmodell.prompt_auto(prompt="What did you do today?")
print(chatmodell.zeige_prompt(chatmodell.gedaechtnis))

## Decoding Parameter

Generieren Sprachmodelle immer dasselbe?
In Folien sollte erklärt werden wie Sampling funktioniert.
Inhalte: Determinismus in LLMs, Temperatur, Nukleus-Sampling

In [ ]:
prompt = "Write a creative metaphor for a quadratic equation."
max_tokens = 64

### Temperatur = 0.0

In [ ]:
for i in range(3):
    chatmodell.prompt(prompt=prompt, max_tokens=max_tokens, temperatur=0.0)

### Temperatur = 0.7

In [ ]:
for i in range(3):
    chatmodell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=0.7)

### Temperatur = 1.8

In [ ]:
for i in range(3):
    chatmodell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=1.8)

### Temperatur & Nukleus-Sampling Visualisiert

Eine kleine interaktive Visualisierung

In [ ]:
from workshop.viz import visualisiere_sampling
visualisiere_sampling(modell=chatmodell)